# Economics, the learning portfolio, and design choice

An experiment withholds dose from a holdout and forgoes outcome. `ValuePerOutcome` converts
outcome units to a numeraire and records where that number came from (a ledger line, rule 4).
Discounting uses `w_t = (1 + rate)^(−t)`; `mid_horizon_factor` is the mean weight. The
opportunity cost is **signed**,

    OC = holdout · dose_per_period · n · mid_horizon_factor · (E[ratio] · value − dose_cost),

so a treatment the prior thinks is net-negative has a negative cost of withholding. The net
value of an experiment is `EVSI − opportunity_cost − fixed_cost`.

In [ ]:
import numpy as np

from axiom.core import Unsupported
from axiom.design import (
    CandidateScore, DecisionSpec, DesignCandidate, EconomicInputs, ExperimentValue,
    LearningPriority, OpportunityCost, Parameter, ProgramSchedule, Recommendation,
    ScheduledExperiment, SensitivityTable, StudySummary, TreatmentCandidate, ValuePerOutcome,
    discount_weights, elasticity, evaluate_candidate, experiment_value, information_value_of,
    mid_horizon_factor, opportunity_cost, pareto_front, perturb, prior_from_history,
    rank_treatments, recommend, schedule_with_cooldown,
)

from axiom.display import enable

enable();  # every axiom result renders itself from here on

In [ ]:
vpo = ValuePerOutcome(value=3.0, outcome_unit="kg", numeraire="USD", source="contract price, 2026 season")
print(vpo.ledger_line().statement)
print("discount weights (8 periods, 2%):", np.round(discount_weights(8, 0.02), 4))
print("mid-horizon factor:", round(mid_horizon_factor(8, 0.02), 4))

In [ ]:
rng = np.random.default_rng(0)
ratio_draws = rng.lognormal(np.log(1.2), 0.3, size=500)  # outcome units per dose unit
oc: OpportunityCost = opportunity_cost(0.25, 8, dose_per_period=20.0, marginal_value_ratio=ratio_draws,
                                      value_per_outcome=vpo, discount_rate=0.02, dose_unit="L", dose_cost_per_unit=1.0)
print(f"dose withheld {oc.dose_withheld:.1f} L (discounted {oc.dose_withheld_discounted:.1f}); outcome forgone {oc.outcome_forgone:.1f} kg")
print(f"opportunity cost {oc.value:.2f} {oc.numeraire}  (ratio {oc.ratio_mean:.3f} ± {oc.ratio_sd:.3f} from {oc.n_ratio_draws} draws)")
negative = opportunity_cost(0.25, 8, 20.0, 0.2, vpo, 0.02, dose_cost_per_unit=1.0)
print("net-negative treatment -> withholding pays:", round(negative.value, 2))

In [ ]:
decision = DecisionSpec(name="continue_dosing", threshold=1.0, value_per_outcome_unit=20000.0, numeraire="USD")
info = information_value_of(decision, prior_mean=1.2, prior_sd=0.5, experiment_se=0.2)
ev: ExperimentValue = experiment_value(info, oc, fixed_cost=250.0)
print(f"information value {ev.information_value:.2f} − opportunity {ev.opportunity_cost:.2f} − fixed {ev.fixed_cost:.2f} = net {ev.net:.2f} {ev.numeraire}")

## Which treatment to learn about next

Past studies are aged with `decayed_sd` and combined by inverse variance into today's prior.
`rank_treatments` scores each candidate's EIG, EVSI and net value; `recommend` is the greedy
knapsack on net value per unit cost, and returns `Unsupported` when nothing is worth running.

In [ ]:
history = [
    StudySummary(treatment="fertilizer", estimate=1.4, se=0.3, periods_ago=12.0, definition="wald"),
    StudySummary(treatment="fertilizer", estimate=1.0, se=0.4, periods_ago=2.0, definition="wald"),
]
mean, sd = prior_from_history(history, half_life_periods=8.0)
print(f"prior from history: {mean:.3f} ± {sd:.3f}")

In [ ]:
candidates = [
    TreatmentCandidate(name="fertilizer", prior_mean=mean, prior_sd=sd, experiment_se=0.2, decision=decision, opportunity_cost=oc.value, fixed_cost=250.0),
    TreatmentCandidate(name="irrigation", prior_mean=0.8, prior_sd=0.6, experiment_se=0.3, decision=decision, opportunity_cost=100.0, fixed_cost=400.0),
    TreatmentCandidate(name="pruning", prior_mean=0.1, prior_sd=0.1, experiment_se=0.3, decision=decision, opportunity_cost=0.0, fixed_cost=60.0),
]
for p in rank_treatments(candidates):
    assert isinstance(p, LearningPriority)
    print(f"#{p.rank} {p.treatment:12s} eig={p.eig:.3f} evsi={p.evoi:8.2f} net={p.net_value:8.2f} cost={p.cost:7.2f}")
rec = recommend(candidates, budget=600.0)
assert isinstance(rec, Recommendation)
print("selected:", rec.selected, "| total net:", round(rec.total_net_value, 2), "| skipped over budget:", rec.detail["skipped_over_budget"] or "none")
print("nothing worth running ->", type(recommend(candidates[2:])).__name__)

## Scoring concrete designs, the Pareto front, and a program schedule

A `DesignCandidate` is one way of running the experiment: a registered method, size, horizon,
holdout share, the standard error it would achieve, its cost and its cooldown.
`evaluate_candidate` scores it on one decision; `pareto_front` keeps the non-dominated
candidates on named objectives (leading `-` means minimize); `schedule_with_cooldown` lays them
end to end, best net value first.

In [ ]:
economics = EconomicInputs(value_per_outcome=vpo, dose_per_period=20.0, discount_rate=0.02, dose_unit="L", dose_cost_per_unit=1.0, marginal_value_ratio=1.2)
designs = [
    DesignCandidate(name="small_holdout", method="difference_in_differences", n_units=20, n_periods=6, holdout_fraction=0.2, experiment_se=0.35, cost=200.0, cooldown_periods=2),
    DesignCandidate(name="large_holdout", method="cluster_based_regression", n_units=60, n_periods=8, holdout_fraction=0.4, experiment_se=0.12, cost=600.0, n_clusters=12, cooldown_periods=4),
    DesignCandidate(name="switchback", method="switchback", n_units=20, n_periods=10, holdout_fraction=0.5, experiment_se=0.2, cost=300.0, cooldown_periods=1),
    DesignCandidate(name="ghost", method="ghost", n_units=40, n_periods=6, holdout_fraction=0.3, experiment_se=0.3, cost=450.0),
]
scores: list[CandidateScore] = [evaluate_candidate(c, decision, prior_mean=mean, prior_sd=sd, economics=economics) for c in designs]
for s in scores:
    print(f"{s.name:14s} eig={s.eig:.3f} evsi={s.evsi:7.2f} oc={s.opportunity_cost:7.2f} cost={s.cost:5.1f} net={s.net_value:8.2f} power={s.power:.3f}")

In [ ]:
front = pareto_front(scores, objectives=("net_value", "-cost", "eig"))
print("Pareto front:", [s.name for s in front])
sched: ProgramSchedule = schedule_with_cooldown(scores, horizon_periods=20)
for slot in sched.slots:
    assert isinstance(slot, ScheduledExperiment)
    print(f"{slot.name:14s} runs [{slot.start:2d}, {slot.end:2d}) free at {slot.free_at:2d}  net {slot.net_value:.2f}")
print("skipped:", sched.skipped, "| total net:", round(sched.total_net_value, 2))

## Sensitivity

`perturb` re-scores every candidate over a grid of one input and records the winner at each
point; a tipping point is a pair of adjacent grid values across which the winner changes.
`elasticity` is `(Δnet / net) / (Δx / x)` at the base point.

In [ ]:
parameter: Parameter = "value_per_outcome"
table: SensitivityTable = perturb(designs, decision, mean, sd, economics, parameter, grid=(0.5, 1.0, 2.0, 3.0, 5.0, 8.0))
for g, w, row in zip(table.grid, table.winners, table.net_values):
    print(f"value={g:4.1f}  winner={w:14s}  nets={tuple(round(v) for v in row)}")
print("base winner:", table.base_winner, "| stable:", table.stable, "| tipping points:", table.tipping_points)
print("elasticities:", {k: round(v, 3) for k, v in elasticity(table).items()})

In [ ]:
se_table = perturb(designs, decision, mean, sd, economics, "experiment_se", grid=(0.5, 0.75, 1.0, 1.5, 2.0))
print(se_table.mode, "-> winners across se multipliers:", se_table.winners)